<a href="https://colab.research.google.com/github/josephineabioye/msc-skincare-reaction-prediction/blob/main/notebooks/03_reaction_label_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
## Notebook Setup Cell

from google.colab import userdata, drive
import os

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_USERNAME = 'josephineabioye'
REPO_NAME = 'msc-skincare-reaction-prediction'

if not os.path.exists(REPO_NAME):
    !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
    %cd {REPO_NAME}
else:
    %cd {REPO_NAME}
    !git pull

!git config user.email "josephineabioye@yahoo.com"
!git config user.name "Josephine Abioye"

drive.mount('/content/drive', force_remount=False)

DRIVE_PROJECT = '/content/drive/MyDrive/msc-skincare-project'
DATA_RAW = f'{DRIVE_PROJECT}/data/raw'
DATA_PROCESSED = f'{DRIVE_PROJECT}/data/processed'

print(f"Working in: {os.getcwd()}")

Cloning into 'msc-skincare-reaction-prediction'...
remote: Enumerating objects: 113, done.
remote: Counting objects: 100% (113/113), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 113 (delta 59), reused 3 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (113/113), 192.42 KiB | 1.96 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/msc-skincare-reaction-prediction/msc-skincare-reaction-prediction
Mounted at /content/drive
Working in: /content/msc-skincare-reaction-prediction/msc-skincare-reaction-prediction


In [4]:
## Load consolidated skincare reviews, remove columns and merge

import spacy
import pandas as pd
from spacy.matcher import PhraseMatcher
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score, f1_score

nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
nlp.add_pipe("sentencizer")

# Load
skincare_reviews = pd.read_parquet(f"{DATA_PROCESSED}/skincare_reviews_english.parquet")

# drop the columns not required
skincare_reviews = skincare_reviews.drop(columns=["Unnamed: 0", "author_id", "lang"], errors="ignore")

# merge review_title + review_text columns
skincare_reviews["review_full"] = (
    (skincare_reviews["review_title"].fillna("") + ". " + skincare_reviews["review_text"].fillna(""))
    .str.replace("\u2019", "'", regex=False)   # curly apostrophe to straight
    .str.replace("\u2018", "'", regex=False)
    .str.strip()
)
print(skincare_reviews.shape)
print("Columns:", skincare_reviews.columns.tolist())

(1089003, 18)
Columns: ['rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color', 'product_id', 'product_name', 'brand_name', 'price_usd', 'review_full']


In [ ]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [6]:
# Lexicons: From the lexicon library created

#Strong Indicators
STRONG = [
    "irritation","irritated","irritating","irritate","burning","burns","burned","burn","stinging","stung","sting","stings","itching","itchy","itchiness","rash","hives",
    "blister","blistering","allergic reaction","allergy","swollen","swelling","welts","chemical burn","contact dermatitis","raw skin","peeled my skin","skin peeled off",
    "skin barrier damaged","damaged my barrier","compromised barrier","barrier damage","skin barrier ruined","damaged my skin","wrecked my barrier","compromised my skin barrier",
    "stripped my skin","stripped all the moisture","over-exfoliated","over exfoliated","sensitized my skin","skin became sensitive","broke me out","broke out","broke out my",
    "caused a breakout","made me break out","gave me acne","gave me pimples","clogs my pores","clogged my pores","clogging my pores","more acne","worse acne","acne got worse",
    "more pimples","hyperpigmentation got worse","dark spots got worse","texture got worse","eyes burned","eyes sting","watery eyes","swollen eyelids","itchy eyes","lip irritation",
    "lip swelling","dried out","dries out","dry out","drying out","dried my skin","dried me out","too drying","very drying","overly drying","so drying","little drying","bit drying",
]

#Dual use terms
DUAL_USE = [
    "acne","pimples","zits","breakout","break outs","break out","breaking out","redness","red patches","red spots","dry","dryness","dehydrated","oily","oiliness","greasy","shiny",
    "sensitive","sensitivity","bumpy","bumps","clogged pores","pore-clogging","pore clogging","congested skin","congestion","congested","flaky","flake","flaking","scaling","peeling",
    "texture","rough","rough texture","purging","tight","tightness","comedones","whiteheads","blackheads","closed comedones","papules","pustules","nodules","cysts","cystic acne",
    "fungal acne","comedogenic","discoloration","hyperpigmentation","dark spots","puffiness","inflamed","inflammation","flare-up","flare up","eczema flare","chapped lips",
    "cracked lips","chapped","deep pimples","painful pimples",
]

#Causal / Worsening phrases that trigger dual use terms
CAUSAL = [
    "caused","made me","made my","left me with","resulted in","after using","since using","every time i use","whenever i use","within hours","immediately after","the next day",
    "triggered","led to","gave me","ended up with","because of","due to","from using","started","start","began","begun",
]

WORSENING = ["worse","worsen","worsened","got worse","aggravate","aggravated","increased","more"]

# Negation cues / words checked in a window before a match
NEG_CUES = {"no","not","never","without","didn't","doesn't","wasn't","isn't","don't","none","cannot","can't","nor","n't","non","hardly","barely"}

# Purpose cues / words checked a match
PURPOSE_CUES = {"for","help","helps","helping","treat","treats","treating","target","targets","soothe","soothes","soothing","calm","calms","calming","reduce","reduces","reducing",
                "prevent","prevents","combat","combats","fight","fights","clear","fade","fades"}

FEAR_CUES = {"afraid","scared","worried","hesitant","nervous","feared","fearful","skeptical"}

# Improvement/Resolution. Checked across the whole sentence (dual-use only).
IMPROVE_PHRASES = [
    "got rid","get rid","gets rid","getting rid","gotten rid","cleared up","clears up","clear up","clearing up","cleared my","calmed down","calm down","calms down","calming down",
    "went away","go away","goes away","going away","died down","settled down","settle down","no longer","under control","helped with","helps with","help with","helped my","helps my",
    "shrink","shrank","shrunk","faded","fading","healed","healing",
]

CONTRAST = {"but","however","though","although","yet","otherwise","except"}

# Adjectives that, when directly before "skin", describes the previous skin situation, not a reaction.
DESCRIPTOR_ADJ = {"oily","dry","sensitive","combination","combo","normal","itchy","dehydrated"}


In [7]:
# Matching and Labelling

def _matcher(terms):
    m = PhraseMatcher(nlp.vocab, attr="LOWER") #Use phrase matching
    m.add("X", list(nlp.pipe(terms)))
    return m

strong_m, dual_m, causal_m, worsen_m = _matcher(STRONG), _matcher(DUAL_USE), _matcher(CAUSAL), _matcher(WORSENING)

def _cue_in_clause(sd, start, cue_set):
    # Check backwards from the matched term for a cue within the same clause.
    # Stop at a contrast word so that cues from another clause do not affect the match.
    for i in range(start - 1, -1, -1):
        w = sd[i].lower_
        if w in CONTRAST:
            break
        if w in cue_set or w.startswith("non-"):
            return True
    return False

def _is_descriptor(sd, start, end):
    # Do not treat phrases describing the skin such as "oily skin" or "occasionally itchy skin" as adverse reactions.
    return end < len(sd) and sd[end].lower_ == "skin" and sd[start].lower_ in DESCRIPTOR_ADJ

def label_review(doc):
    for sent in doc.sents:
        sd = sent.as_doc()
        # Check whether the sentence is framed as an improvement.
        improved = any(p in sd.text.lower() for p in IMPROVE_PHRASES)

        # STRONG TERMS: trigger a reaction label unless they are negated, expressed as a fear/concern, or are only describing the skin.
        for _, start, end in strong_m(sd):
            if _cue_in_clause(sd, start, NEG_CUES):   continue
            if _cue_in_clause(sd, start, FEAR_CUES):  continue
            if _is_descriptor(sd, start, end):        continue
            return 1

        # DUAL-USE TERMS: require a causal or worsening cue in the same sentence. They are also suppressed when they are descriptors, negated, used to
        # describe the product's purpose, expressed as a fear/concern, or when the whole sentence is framed around improvement.
        dual = dual_m(sd)
        if dual and (causal_m(sd) or worsen_m(sd)) and not improved:
            for _, start, end in dual:
                if _is_descriptor(sd, start, end):        continue
                if _cue_in_clause(sd, start, NEG_CUES):   continue
                if _cue_in_clause(sd, start, PURPOSE_CUES): continue
                if _cue_in_clause(sd, start, FEAR_CUES):  continue
                return 1
    return 0

In [8]:
# Confirmation
sample = skincare_reviews.sample(5000, random_state=42).copy()
sample["reaction_label"] = [label_review(doc) for doc in nlp.pipe(sample["review_full"].fillna(""), batch_size=200)]
print(sample["reaction_label"].value_counts(normalize=True))

reaction_label
0    0.8932
1    0.1068
Name: proportion, dtype: float64


In [9]:
# validation set for self labelling
to_annotate = skincare_reviews.sample(200, random_state=2026)[["review_full"]].copy()
to_annotate["manual_label"] = ""
to_annotate.to_csv(f"{DATA_PROCESSED}/to_annotate.csv", index=False)
print("Validation set ready for download")


Validation set ready for download


In [10]:
# Last Kappa Calculation
val = pd.read_csv(f"{DATA_PROCESSED}/annotations.csv")
val["manual_label"] = val["manual_label"].astype(int)
val["model_label"]  = val["review_full"].apply(lambda t: label_review(nlp(t)))

k  = cohen_kappa_score(val["manual_label"], val["model_label"])
p  = precision_score(val["manual_label"], val["model_label"], zero_division=0)
r  = recall_score(val["manual_label"], val["model_label"], zero_division=0)
f1 = f1_score(val["manual_label"], val["model_label"], zero_division=0)

print(f"Rows: {len(val)}")
print(f"Cohen's kappa: {k:.3f}")
print(f"Precision: {p:.3f}   Recall: {r:.3f}   F1: {f1:.3f}")
print(pd.crosstab(val["manual_label"], val["model_label"], rownames=["you"], colnames=["model"]))

# Disagreements, for the write-up
fp = val[(val["manual_label"] == 0) & (val["model_label"] == 1)]
fn = val[(val["manual_label"] == 1) & (val["model_label"] == 0)]
print(f"\nFALSE POSITIVES ({len(fp)})"); [print(" •", t[:300]) for t in fp["review_full"]]
print(f"\nFALSE NEGATIVES ({len(fn)})"); [print(" •", t[:300]) for t in fn["review_full"]]

Rows: 200
Cohen's kappa: 0.610
Precision: 0.583   Recall: 0.737   F1: 0.651
model    0   1
you           
0      171  10
1        5  14

FALSE POSITIVES (10)
 • . i really enjoyed this moisturizer. i loved the texture and it made my skin feel smooth and nourished. i really liked the smell too. it was fruity and pleasant but not overbearing which i really like. i've only used it a couple days but my skin looks really good!
 • Rejuvenated Youthful Glow. In just under two-weeks, Dior Capture Totale Eye Serum took my saggy, baggy eyes from dull and tired to radiant and beaming. I've tried many eye serums over the past several years, and I can honestly say that Dior Capture Totale Eye Serum works its magic within just a few 
 • Noticeable results with acne scars. First of all, I do not write reviews but felt I must extol the praises of this magical oil in case there's another person out there in a similar situation as myself. I'm a 39 year old woman who has dealt with cystic acne since I wa

[None, None, None, None, None]

In [12]:
# Apply the label across the full reviews
from tqdm.auto import tqdm

texts = skincare_reviews["review_full"].fillna("").tolist()
labels = [label_review(doc) for doc in tqdm(nlp.pipe(texts, batch_size=1000), total=len(texts))]

skincare_reviews["reaction_label"] = labels
print(skincare_reviews["reaction_label"].value_counts(normalize=True))
skincare_reviews.to_parquet(f"{DATA_PROCESSED}/skincare_reviews_labelled.parquet", index=False)
print("Saved skincare_reviews_labelled.parquet")

  0%|          | 0/1089003 [00:00<?, ?it/s]

reaction_label
0    0.889392
1    0.110608
Name: proportion, dtype: float64
Saved skincare_reviews_labelled.parquet


In [ ]:
#Round 4 Kappa Calculation result

import pandas as pd
from sklearn.metrics import cohen_kappa_score, precision_score, recall_score, f1_score

ann = pd.read_csv(f"{DATA_PROCESSED}/annotations.csv", index_col=0)

# sanity check first
print("Columns:", ann.columns.tolist())
print(ann["manual_label"].value_counts(dropna=False), "\n")

m = ann.dropna(subset=["manual_label"]).copy()
m["manual_label"] = m["manual_label"].astype(int)

# recompute model labels with CURRENT rules
m["model_label"] = [label_review(doc) for doc in nlp.pipe(m["review_full"].fillna(""), batch_size=100)]

print("Rows compared:", len(m), "\n")
print("Cohen's kappa:", round(cohen_kappa_score(m["manual_label"], m["model_label"]), 3))
print("Precision:", round(precision_score(m["manual_label"], m["model_label"]), 3))
print("Recall:   ", round(recall_score(m["manual_label"], m["model_label"]), 3))
print("F1:       ", round(f1_score(m["manual_label"], m["model_label"]), 3))

print("\nConfusion (rows = your label, cols = model):")
print(pd.crosstab(m["manual_label"], m["model_label"], rownames=["you"], colnames=["model"]))

print("\nDisagreements")
for _, r in m[m["manual_label"] != m["model_label"]].iterrows():
    tag = "FALSE POS" if r["model_label"] == 1 else "FALSE NEG"
    print(f"[{tag}] {str(r['review_full'])[:160]}")

Columns: ['review_full', 'manual_label']
manual_label
0    182
1     18
Name: count, dtype: int64 

Rows compared: 200 

Cohen's kappa: 0.593
Precision: 0.647
Recall:    0.611
F1:        0.629

Confusion (rows = your label, cols = model):
model    0   1
you           
0      176   6
1        7  11

Disagreements
[FALSE POS] . I was not very impressed. My skin type is very dry and this moisturizer is just way too lightweight for my skin type. It soaked right in and almost made my fa
[FALSE NEG] Purchased this twice and love it. This is a very nourishing and hydrating toner. I use it in the evening after washing my face. You have to let your skin adjust
[FALSE POS] . I was gifted this product by Glow Recipe. I tried out this product this morning after washing my face and my skin has never felt so soft and plump. Certain pr
[FALSE POS] Just not sure.... I've been using this for about 2 weeks now... I haven't really noticed any reduction in redness. It smells great and hasn't caused any ir

In [ ]:
# Display examples of reviews classified as reactions (label = 1) and non-reactions (label = 0)
import textwrap

print("labelled REACTION (1)")
for t in sample[sample.reaction_label == 1]["review_full"].sample(15, random_state=1):
    print("•", textwrap.shorten(t, 200))

print("\nlabelled NO reaction (0)")
for t in sample[sample.reaction_label == 0]["review_full"].sample(15, random_state=1):
    print("•", textwrap.shorten(t, 200))

labelled REACTION (1)
• 1 OF THE WORST PRODUCTS I EVER TRIED!!!!. This product made my acne worse I have very bad painful, big,red inflamed under the skin blemishes and tons upon tons of white heads and tiny bumps on [...]
• Extra moisture!. I am ecstatic that I was able to try this free for review purposes. I think I have found a new addition to my skin care line up. I have consistently been using this everyday [...]
• It's got Retinol in it.. This stuff feels weird, takes a long time to absorb, and sometimes it stings but it must be doing something. My skin is a bit brighter and my red marks have faded a bit.
• . I have ruled out toner from my routine almost 2 year ago due to having super sensitive skin. I've tried many brand name toners and literally anything recommended to me gave me awful allergies [...]
• Helped so much with rosacea. I have very fair, very fickle, pink/red skin. I have flushing rosacea and get flare ups A LOT, which was resulting in tiny, irritated rosacea “bumps

In [ ]:
# Identify and display the first rule trigger responsible for classifying a review as a reaction.
def first_trigger(doc):
    for sent in doc.sents:
        sd = sent.as_doc()
        for _, s, e in strong_m(sd):
            if not _suppressed(sd, s):
                return ("STRONG", sd[s:e].text, sent.text)
        dual = dual_m(sd)
        if dual and (causal_m(sd) or worsen_m(sd)):
            for _, s, e in dual:
                if not _suppressed(sd, s):
                    return ("DUAL", sd[s:e].text, sent.text)
    return None

for t in sample[sample.reaction_label == 1]["review_full"].sample(25, random_state=2):
    trig = first_trigger(nlp(t))
    if trig:
        print(f"[{trig[0]}] «{trig[1]}»  ⟵  {trig[2][:130]}")

[STRONG] «break out»  ⟵  Normally when I use a new cleanser, I break out in either hives or acne, so I was a little scared to try this.
[STRONG] «stings»  ⟵  It stings your eyes not only with the worst pain imaginable, but for the longest period of time.
[STRONG] «break out»  ⟵  Normally my skin will break out when I try new products on my face but with this product it hasn't broken out any yet!
[STRONG] «Broke out»  ⟵  Broke out.
[STRONG] «make me break out»  ⟵  I immediately noticed a strong sun screen scent (which I hate because most sun screens make me break out).
[DUAL] «dries»  ⟵  Since it has charcoal in it, sometimes I felt like it would make my skin look a little dirty after it dries, but I have started to
[DUAL] «shiny»  ⟵  However Clinique knocked my socks off with this gel moisturizer that keeps my skin hydrated in the cold weather seasons when I rea
[DUAL] «breakouts»  ⟵  My skin is usually fairly clear, but I have been prone to occasional breakouts due to hormonal and die

In [ ]:
val = pd.read_csv(f"{DATA_PROCESSED}/annotations.csv")
val["model_label"] = val["review_full"].apply(lambda t: label_review(nlp(t)))

fp = val[(val["manual_label"] == 0) & (val["model_label"] == 1)]
fn = val[(val["manual_label"] == 1) & (val["model_label"] == 0)]

print(f"FALSE POSITIVES ({len(fp)})  — engine said 1, you said 0")
for t in fp["review_full"]:
    print(" •", t[:350])

print(f"\nFALSE NEGATIVES ({len(fn)})  — engine said 0, you said 1")
for t in fn["review_full"]:
    print(" •", t[:350])

FALSE POSITIVES (16)  — engine said 1, you said 0
 • cleared my little bumps. so this is the first time I am using the peeling solution, and it already made a huge difference on my skin texture and oiliness. I have combination skin, but I get super oily in my t zone area, while my cheeks are fairly dry. because of my oiliness, my little bumps on my forehead are more prone to becoming bigger and more 
 • refreshed. I have extremely dry occasionally itchy skin and this mask had Made me see great improvement in the hydration and appearance of my skin. After using it daily my skin looks very dewy and youthful without being oily. When you put it on it is very refreshing and instantly cooling. Highly recommend this mask.
 • Serum smells terrible!. Received this as a sample...had to wash it off right away. Very strong smell and felt irritating. Would not purchase.
 • Dry. I love everything from fresh specially their regular lip balm. I was so excited when they came out the balm with colors bu


## VALIDATION ITERATION LOG

| Round | Change made and reason | Positive rate (5k sample) | Kappa |
|---|---|---:|---:|
| 1 | The initial tiered rules were applied; Strong terms could trigger a reaction label on their own, with negation and purpose/indication suppression  to reduce the false positives. Dual-use terms required a causal or worsening cue in the same sentence. The review title was also merged with the review body so that all available text was considered during labelling. | 9.72% | Not measured |
| 2 | Noticed that curly apostrophes were preventing some negation cues from matching correctly, so apostrophes were normalised. Observed that terms such as "puffiness", "inflamed"/"inflammation", "flare-up", "eczema flare", and "chapped"/"cracked lips" could be indication-related rather than direct adverse reactions, so they were moved to the dual-use category. Identified that some reaction phrases were being missed due to variations in wording, so base verb forms (burn, sting, irritate) and active clog-related phrases were added to improve recall. | 9.18% | Not measured |
| 3 | Noticed that some negation patterns and purpose-related descriptions were still being incorrectly identified as reactions, so “non” was added to the negation cues and -ing purpose forms (soothing, calming, reducing) were included. Observed that skin-type descriptions such as “oily skin”, “dry skin”, and “sensitive skin” were triggering false positives, so a skin-type suppressor was added. Identified that some breakout-related reactions were being missed due to variations in wording, so “break out” and “broke out” were added to improve recall. The remaining false positives were left for the transformer fallback and manual validation. | 9.06% | Not measured |
| 4 | The first blind validation was carried out against a manually labelled reference set of 200 reviews, with no rule changes made at this stage. Error analysis showed that some of the remaining missed reactions were caused by different word forms, including singular/plural and other word variations. | 9.06% (remained unchanged from R3) | 0.594 |
| 5 | Observed that the remaining missed reactions could be better captured by matching different word forms, so the PhraseMatcher was changed from surface-form matching to lemma matching. However, validation on a fresh 200-review set showed that this caused many false positives. In particular, lemma matching caused the strong phrase “broke out” to overlap with the dual-use term “break out”, resulting in matches for habitual and hypothetical situations. The approach was therefore rejected. | 14.02% | 0.225 |
| 6 | As lemma matching resulted in a substantial drop in precision, the PhraseMatcher was changed back to surface-form matching. The same fresh validation set was re-measured, and the results showed that a precision problem still remained in the base rules. This set was therefore retained as a development set for further error analysis rather than used as the final validation set. | Not re-measured on 5k | 0.285 |
| 7 | Errors identified from the development set were used for refinements. Sentence-wide improvement suppression was added to prevent positive treatment outcomes from being labelled as reactions. The 5-word negation window was replaced with clause-aware negation so that cues were more closely linked to the matched term. Fear and hypothetical mentions were also suppressed, the skin-type descriptor suppression was expanded, and additional product-caused dryness terms and onset-related causal cues were added to improve recall. | 10.68% | Validated in R8 |
| 8 | A final blind validation was carried out on a fresh held-out set of 200 reviews. The classifier achieved a precision of 0.58, recall of 0.74, and F1-score of 0.65. Following this validation, the final labelling rules were applied to all English-language reviews, which resulted in a positive rate of 11.06%. | 11.06% (on all reviews) | 0.610 |

**Note:** The earlier Kappa value (Kappa = 0.594) is not directly comparable with the final result. The Round 8 result (Kappa = 0.610) is the final validation figure, as it was measured using the finalised annotation rubric on a held-out sample.